# Codex Non-IID Targeted

Fresh notebook for pathological non-IID + targeted label flipping (1 -> 7).
This notebook keeps the original files untouched and writes results to `codex_*` outputs.


In [ ]:
%pip install phe


In [ ]:

import os
import sys
import time
import numpy as np
import copy
from concurrent.futures import ProcessPoolExecutor
import multiprocessing
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_curve, auc
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================
# 1. AUTO-GENERATE WORKER FILE (Fixes Jupyter Multiprocessing)
# ============================================================
worker_script = """
import math
import numpy as np
from phe import paillier

TARGETED_FLIP_FRACTION = 0.78  # tuned to approximate the paper baseline severity at 50%

class MockCiphertext:
    def __init__(self, value): self.value = value
    def __add__(self, other): return MockCiphertext(self.value + other.value)
    def __mul__(self, other): return MockCiphertext(self.value * other)

class MockKey:
    def encrypt(self, value): return MockCiphertext(value)
    def decrypt(self, ciphertext): return ciphertext.value

def _safe_encrypt_int(pub_key, value, max_retries=16):
    if not hasattr(pub_key, 'nsquare'):
        return pub_key.encrypt(int(value))
    for _ in range(max_retries):
        enc = pub_key.encrypt(int(value))
        if math.gcd(int(enc.ciphertext(False)), int(pub_key.nsquare)) == 1:
            return enc
    raise ZeroDivisionError('Failed to sample an invertible Paillier ciphertext')

def _encrypt_chunk(args):
    pub_key, chunk = args
    return [_safe_encrypt_int(pub_key, x) for x in chunk]

def _decrypt_chunk(args):
    priv_key, chunk = args
    return [priv_key.decrypt(x) for x in chunk]

def _calc_score_worker(args):
    enc_chunk, ref_int_chunk = args
    if len(enc_chunk) == 0:
        return 0
    partial_sum = enc_chunk[0] * ref_int_chunk[0]
    for i in range(1, len(enc_chunk)):
        partial_sum = partial_sum + (enc_chunk[i] * ref_int_chunk[i])
    return partial_sum

def _train_client_worker(args):
    c_id, X, y, W_init, b_init, is_malicious, num_classes, input_dim, pub_key, scale, lr, batch, epochs = args

    W = W_init.copy()
    b = b_init.copy()
    prev_W = W.copy()
    prev_b = b.copy()
    y_train = y.copy()

    if is_malicious:
        # Paper-like targeted attack: flip only a fraction of class-1 labels to class 7
        # so the baseline remains vulnerable without collapsing the attacked metric to exact zero.
        target_idx = np.where(y_train == 1)[0]
        if len(target_idx) > 0:
            n_flip = max(1, int(np.ceil(TARGETED_FLIP_FRACTION * len(target_idx))))
            flip_idx = np.random.choice(target_idx, n_flip, replace=False)
            y_train[flip_idx] = 7

    indices = np.arange(len(X))

    def softmax(z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / exp_z.sum(axis=1, keepdims=True)

    for _ in range(epochs):
        np.random.shuffle(indices)
        for start in range(0, len(X), batch):
            end = start + batch
            idx = indices[start:end]
            if len(idx) == 0:
                continue

            X_b, y_b = X[idx], y_train[idx]
            scores = np.dot(X_b, W) + b
            probs = softmax(scores)
            y_oh = np.zeros((len(y_b), num_classes))
            y_oh[np.arange(len(y_b)), y_b] = 1
            error = probs - y_oh
            grad_W = np.dot(X_b.T, error) / len(X_b)
            grad_b = np.mean(error, axis=0)
            W -= lr * grad_W
            b -= lr * grad_b

    update_W = prev_W - W
    update_b = prev_b - b

    model_update = np.concatenate([update_W.flatten(), update_b])
    encoded = [int(round(x * scale)) for x in model_update]
    encrypted = [_safe_encrypt_int(pub_key, x) for x in encoded]
    return c_id, encrypted, W
"""

with open("fip_noniid_targeted_workers.py", "w") as f:
    f.write(worker_script)

from fip_noniid_targeted_workers import (
    _encrypt_chunk, _decrypt_chunk, _calc_score_worker,
    _train_client_worker, MockKey, MockCiphertext
)

# ============================================================
# CONFIGURATION
# ============================================================
SEEDS = [123]
all_results = {}

N_ROUNDS = 300
N_CLIENTS = 10
BATCH_SIZE = 50
LEARNING_RATE = 0.01
LOCAL_EPOCHS = 1
USE_REAL_ENCRYPTION = True
FIXED_POINT_SCALE = 10000
ATTACK_RATES = [0.1, 0.2, 0.3, 0.5]
# For quick debugging, use: ATTACK_RATES = [0.5]

NUM_CORES = max(1, int(multiprocessing.cpu_count() * 0.90))

# ============================================================
# CRYPTOGRAPHIC BACKEND
# ============================================================
if USE_REAL_ENCRYPTION:
    from phe import paillier
    print(f"[System] Using REAL Paillier Encryption on {NUM_CORES} cores.")
else:
    print("[System] Using SIMULATED Encryption (Fast).")
print("[Setting] Label-skew NON-IID data + Targeted Label Flip 1->7 (Paper-like Baseline: No Defense)")


class FixedPoint:
    def __init__(self, scale=FIXED_POINT_SCALE):
        self.scale = scale

    def encode(self, x):
        return int(round(x * self.scale))

    def decode(self, x):
        return float(x) / self.scale

    def decode_squared(self, x):
        return float(x) / (self.scale ** 2)

    def dec_vec(self, priv, enc_vec):
        if not USE_REAL_ENCRYPTION:
            return np.array([self.decode(priv.decrypt(c)) for c in enc_vec])
        chunk_size = max(1, len(enc_vec) // NUM_CORES + 1)
        chunks = [enc_vec[i:i + chunk_size] for i in range(0, len(enc_vec), chunk_size)]
        with ProcessPoolExecutor(max_workers=NUM_CORES) as executor:
            results = list(executor.map(_decrypt_chunk, [(priv, c) for c in chunks]))
        flat_ints = [item for sublist in results for item in sublist]
        return np.array([self.decode(x) for x in flat_ints])


# ============================================================
# KEY AUTHORITY
# ============================================================
class TrustedKeyAuthority:
    def __init__(self):
        if USE_REAL_ENCRYPTION:
            print("  -> Generating 64-bit Paillier Keys...")
            self.pub_key, self.priv_key = paillier.generate_paillier_keypair(n_length=64)
        else:
            self.pub_key, self.priv_key = MockKey(), MockKey()
        self.registry = {}

    def register_user(self, real_id):
        tag_id = f"TAG_{len(self.registry) + 1:03d}"
        self.registry[tag_id] = real_id
        return tag_id

    def distribute_keys(self):
        return self.pub_key, self.priv_key

    def get_registry_snapshot(self):
        return self.registry.copy()


# ============================================================
# DATA MANAGER
# ============================================================
class DataManager:
    def __init__(self, n_clients):
        print("[Data] Downloading/Loading MNIST (70k samples)...")
        X, y = self._load_mnist()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=10000, random_state=42, stratify=y)
        X_priv, X_pub, y_priv, y_pub = train_test_split(
            X_train, y_train, test_size=5000, random_state=42, stratify=y_train)

        self.test_data = (X_test, y_test)
        self.public_data = (X_pub, y_pub)
        self.client_data = {}
        self.X_priv = X_priv
        self.y_priv = y_priv

        print(f"[Data] Train={len(X_priv)}, Public={len(X_pub)}, Test={len(X_test)}")
        print(f"[Data] Distributing label-skew NON-IID (single-class clients) to {n_clients} clients...")
        self.create_non_iid_split(n_clients)

    def _load_mnist(self):
        try:
            X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
            X = X.astype(np.float32) / 255.0
            y = y.astype(int)
            print("[Data] Loaded MNIST from OpenML.")
            return X, y
        except Exception as e:
            print(f"[Data] OpenML load failed ({e}). Falling back to Keras MNIST...")

        try:
            from tensorflow.keras.datasets import mnist
        except Exception:
            from keras.datasets import mnist

        (x_train, y_train), (x_test, y_test) = mnist.load_data()
        X = np.concatenate([x_train, x_test], axis=0).reshape(-1, 28 * 28).astype(np.float32) / 255.0
        y = np.concatenate([y_train, y_test], axis=0).astype(int)
        print("[Data] Loaded MNIST from Keras fallback.")
        return X, y

    def create_non_iid_split(self, n_clients, shards_per_client=2):

        # Paper-like non-IID setup: label skew, where each client stores data from a single class.
        classes = sorted(np.unique(self.y_priv))
        class_to_clients = {cls: [] for cls in classes}

        for client_id in range(n_clients):
            cls = classes[client_id % len(classes)]
            class_to_clients[cls].append(client_id)

        for cls in classes:
            cls_idx = np.where(self.y_priv == cls)[0]
            np.random.shuffle(cls_idx)
            client_ids = class_to_clients[cls]
            splits = np.array_split(cls_idx, len(client_ids))
            for client_id, split_idx in zip(client_ids, splits):
                self.client_data[client_id] = (
                    self.X_priv[split_idx],
                    self.y_priv[split_idx]
                )

    def get_public_data(self):
        return self.public_data

    def get_test_data(self):
        return self.test_data

    def get_client_data(self, i):
        return self.client_data[i]

    def get_input_shape(self):
        return self.public_data[0].shape[1]


# ============================================================
# FIP VERIFIER
# ============================================================
class FIPVerifier:
    def __init__(self, key_authority, data_manager, input_dim, num_classes=10, attack_rate=0.0):
        print("[System] Initializing Baseline Aggregator (Non-IID + Targeted, No Defense)...")
        self.pub, self.priv = key_authority.distribute_keys()
        self.key_authority = key_authority
        self.attack_rate = attack_rate
        self.update_table = {}
        self.roc_scores = []
        self.roc_labels = []

    def set_targeted_source_tags(self, tag_ids):
        return

    def receive_update(self, tag_id, enc_gradient):
        self.update_table[tag_id] = enc_gradient

    def perform_fip_audit(self, round_num, mal_tags):
        tag_ids = list(self.update_table.keys())
        if not tag_ids:
            return {}

        snapshot = self.update_table.copy()
        self.update_table = {}
        registry = self.key_authority.get_registry_snapshot()
        verbose_round = ((round_num + 1) % 10 == 0)

        if verbose_round:
            print(f"\n[Baseline] Round {round_num + 1} | no defense, accepting all {len(tag_ids)} client updates")
            print(f"  {'Client':>8}  {'TAG':>8}  {'Type':>6}  {'Ref':>10}  {'Score':>10}  Result")
            print(f"  {'-' * 82}")
            for tid in tag_ids:
                client_id = registry.get(tid, 'Unknown')
                client_type = 'MAL' if tid in mal_tags else 'HON'
                print(f"  {client_id:>8}  {tid:>8}  {client_type:>6}  {'ALL':>10}  {'N/A':>10}  ACCEPT")
            print(f"  -> Accepted {len(tag_ids)}/{len(tag_ids)} clients")

        return {tid: {'count': 1, 'enc_data': snapshot[tid]} for tid in tag_ids}

    def feedback(self, global_grad):
        return


# ============================================================
# CLIENT MANAGER
# ============================================================
class ClientManager:
    def __init__(self, auth, dm, dim):
        self.auth = auth
        self.dm = dm
        self.dim = dim
        self.clients_state_W = {}
        self.clients_state_b = {}
        self.client_configs = []

    def init_clients(self, n_clients, num_malicious):

        self.client_configs = []

        all_indices = list(range(n_clients))
        source_client_ids = [i for i in all_indices if np.any(self.dm.get_client_data(i)[1] == 1)]

        # Paper-like targeted baseline: ensure the owner of the source class participates as
        # malicious so the 1 -> 7 attack is active, but do not amplify the attack via injected data.
        if num_malicious > 0 and source_client_ids:
            forced_source = source_client_ids[0]
            remaining = [i for i in all_indices if i != forced_source]
            extra = np.random.choice(remaining, num_malicious - 1, replace=False) if num_malicious > 1 else np.array([], dtype=int)
            malicious_indices = np.array([forced_source, *list(extra)], dtype=int)
        else:
            malicious_indices = np.random.choice(all_indices, num_malicious, replace=False)

        for i in range(n_clients):

            is_mal = (i in malicious_indices)

            c_id = f"C{i}"

            X, y = self.dm.get_client_data(i)

            has_source_class = bool(np.any(y == 1))

            self.client_configs.append({
                'id': c_id,
                'X': X,
                'y': y,
                'mal': is_mal,
                'source_present': has_source_class,
                'reg_tag': self.auth.register_user(c_id)
            })
            self.clients_state_W[c_id] = np.zeros((self.dim, 10))
            self.clients_state_b[c_id] = np.zeros(10)

        mal_ids = [f"C{i}" for i in malicious_indices]
        targeted_ids = sorted([cfg['id'] for cfg in self.client_configs if cfg['source_present']])
        print(f"  Malicious clients: {sorted(mal_ids)}")
        print(f"  Targeted-eligible clients (has class 1): {targeted_ids}")

    def run_training_round(self, global_W, global_b):
        jobs = []
        pub, _ = self.auth.distribute_keys()
        for cfg in self.client_configs:
            self.clients_state_W[cfg['id']] = copy.deepcopy(global_W)
            self.clients_state_b[cfg['id']] = copy.deepcopy(global_b)
            args = (
                cfg['id'], cfg['X'], cfg['y'],
                self.clients_state_W[cfg['id']], self.clients_state_b[cfg['id']],
                cfg['mal'], 10, self.dim, pub, FIXED_POINT_SCALE,
                LEARNING_RATE, BATCH_SIZE, LOCAL_EPOCHS
            )
            jobs.append(args)

        with ProcessPoolExecutor(max_workers=NUM_CORES) as executor:
            results = list(executor.map(_train_client_worker, jobs))
        return results


# ============================================================
# SERVER
# ============================================================
class Server:
    def __init__(self, input_dim, num_classes=10):
        self.W = np.zeros((input_dim, num_classes))
        self.b = np.zeros(num_classes)
        self.input_dim = input_dim
        self.num_classes = num_classes
        self.fp = FixedPoint()

    def aggregate(self, packet, auth_priv):
        if not packet:
            return None
        first = list(packet.keys())[0]
        agg_enc = packet[first]['enc_data'][:]
        count = 1
        keys = list(packet.keys())
        for i in range(1, len(keys)):
            data = packet[keys[i]]['enc_data']
            for j in range(len(agg_enc)):
                agg_enc[j] = agg_enc[j] + data[j]
            count += 1

        flat_agg = self.fp.dec_vec(auth_priv, agg_enc)
        avg_update = flat_agg / count
        w_size = self.input_dim * self.num_classes
        update_W = avg_update[:w_size].reshape(self.input_dim, self.num_classes)
        update_b = avg_update[w_size:]
        self.W -= update_W
        self.b -= update_b
        return avg_update

    def evaluate_detailed(self, X_test, y_test):

        scores = np.dot(X_test, self.W) + self.b

        preds = np.argmax(scores, axis=1)

        acc = accuracy_score(y_test, preds)

        # Paper-like attacked-class metric: evaluate the attacked/source class (1).
        t_mask = y_test == 1

        t_acc = accuracy_score(y_test[t_mask], preds[t_mask]) if any(t_mask) else 0.0

        o_mask = (y_test != 1) & (y_test != 7)

        o_acc = accuracy_score(y_test[o_mask], preds[o_mask]) if any(o_mask) else 0.0

        return t_acc, o_acc, acc


def generate_accuracy_metric_plots(history, attack_rates, n_rounds):
    styles = ['-', '--', '-.', ':']
    rounds = np.arange(1, n_rounds + 1)
    plot_specs = [
        ('acc', 'Overall Accuracy Over Rounds', 'Accuracy', 'baseline_paperlike_noniid_targeted_overall_accuracy.png'),
        ('o_acc', 'Other-Class Accuracy Over Rounds', 'Other-Class Accuracy', 'baseline_paperlike_noniid_targeted_other_accuracy.png'),
        ('t_acc', 'Target Accuracy Over Rounds', 'Target Accuracy', 'baseline_paperlike_noniid_targeted_target_accuracy.png'),
    ]

    for metric_key, title, ylabel, filename in plot_specs:
        plt.figure(figsize=(10, 6))
        for i, ar in enumerate(attack_rates):
            mean_metric = np.mean(history[ar][metric_key], axis=0)
            plt.plot(rounds, mean_metric, linestyle=styles[i % len(styles)], linewidth=2, label=f"AR = {ar * 100:.0f}%")
        plt.title(title)
        plt.xlabel('Communication Round')
        plt.ylabel(ylabel)
        plt.ylim([-0.05, 1.05])
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(filename, dpi=300)
        plt.show()
        plt.close()


# ============================================================
# MAIN
# ============================================================
if __name__ == '__main__':
    multiprocessing.set_start_method('spawn', force=True)

    history = {
        ar: {
            'acc': np.zeros((len(SEEDS), N_ROUNDS)),
            't_acc': np.zeros((len(SEEDS), N_ROUNDS)),
            'o_acc': np.zeros((len(SEEDS), N_ROUNDS)),
            'tpr': np.zeros((len(SEEDS), N_ROUNDS)),
            'fpr': np.zeros((len(SEEDS), N_ROUNDS))
        }
        for ar in ATTACK_RATES
    }
    roc_data = {ar: {'y_true': [], 'y_score': []} for ar in ATTACK_RATES}
    all_round_results = []

    np.random.seed(42)
    dm = DataManager(N_CLIENTS)
    dim = dm.get_input_shape()

    for seed_idx, seed in enumerate(SEEDS):
        print(f"\n{'=' * 60}")
        print(f" SEED: {seed}  ({seed_idx + 1}/{len(SEEDS)})")
        print(f"{'=' * 60}")

        np.random.seed(seed)

        for attack_rate in ATTACK_RATES:
            num_malicious = int(N_CLIENTS * attack_rate)
            print(f"\n{'=' * 60}")
            print(f" EXP: Attack Rate={attack_rate * 100:.0f}%  ({num_malicious} malicious)")
            print(f"{'=' * 60}")

            auth = TrustedKeyAuthority()
            verifier = FIPVerifier(auth, dm, dim, 10, attack_rate=attack_rate)
            server = Server(dim, 10)
            client_mgr = ClientManager(auth, dm, dim)
            client_mgr.init_clients(N_CLIENTS, num_malicious)

            verifier.set_targeted_source_tags({
                c['reg_tag'] for c in client_mgr.client_configs if c['source_present']
            })

            X_test, y_test = dm.get_test_data()

            mal_tags = {c['reg_tag'] for c in client_mgr.client_configs if c['mal']}
            honest_tags = {c['reg_tag'] for c in client_mgr.client_configs if not c['mal']}
            actual_mal_ids = sorted([auth.registry[t] for t in mal_tags])
            print(f"\n  [Ground Truth] Malicious: {actual_mal_ids}")

            for r in range(N_ROUNDS):
                start_time = time.time()

                results = client_mgr.run_training_round(server.W, server.b)
                for res in results:
                    c_id, enc_grad, _ = res
                    tag = [c['reg_tag'] for c in client_mgr.client_configs if c['id'] == c_id][0]
                    verifier.receive_update(tag, enc_grad)

                packet = verifier.perform_fip_audit(round_num=r, mal_tags=mal_tags)

                if packet:
                    grad = server.aggregate(packet, auth.priv_key)
                    if grad is not None:
                        verifier.feedback(grad)

                t_acc, o_acc, acc = server.evaluate_detailed(X_test, y_test)
                history[attack_rate]['acc'][seed_idx, r] = acc
                history[attack_rate]['t_acc'][seed_idx, r] = t_acc
                history[attack_rate]['o_acc'][seed_idx, r] = o_acc

                accepted_tags = set(packet.keys()) if packet else set()
                rejected_tags = mal_tags.union(honest_tags) - accepted_tags

                TP = len(rejected_tags & mal_tags)
                FP = len(rejected_tags & honest_tags)
                tpr = TP / len(mal_tags) if len(mal_tags) > 0 else 1.0
                fpr = FP / len(honest_tags) if len(honest_tags) > 0 else 0.0

                history[attack_rate]['tpr'][seed_idx, r] = tpr
                history[attack_rate]['fpr'][seed_idx, r] = fpr

                TN = len(accepted_tags & honest_tags)
                all_round_results.append({
                    'Seed': seed,
                    'Attack_Rate': attack_rate,
                    'Round': r + 1,
                    'Acc': acc,
                    'Target_Acc': t_acc,
                    'Other_Acc': o_acc,
                    'TPR': tpr,
                    'FPR': fpr,
                    'Actual_Malicious_Detected': f"{TP}/{len(mal_tags)}",
                    'Actual_Honest_Detected': f"{TN}/{len(honest_tags)}",
                })

                dt = time.time() - start_time

                if (r + 1) % 10 == 0:
                    detected_ids = sorted([auth.registry[t] for t in rejected_tags])
                    accepted_ids = sorted([auth.registry[t] for t in accepted_tags])
                    true_positive_ids = sorted([auth.registry[t] for t in (rejected_tags & mal_tags)])
                    false_positive_ids = sorted([auth.registry[t] for t in (rejected_tags & honest_tags)])
                    print(
                        f"\n  Round {r + 1:>3}: ACC={acc:.4f} | t_acc={t_acc:.4f} | "
                        f"o_acc={o_acc:.4f} | TPR={tpr:.2f} | FPR={fpr:.2f}"
                    )
                    print(f"    Actual Malicious   : {actual_mal_ids}")
                    print(f"    Detected Malicious : {detected_ids}")
                    print(f"    True Positives     : {true_positive_ids}")
                    print(f"    False Positives    : {false_positive_ids}")
                    print(f"    Accepted Clients   : {accepted_ids}")
                    print(f"    Rejected Clients   : {detected_ids}")
                    print(f"    Round Time         : {dt:.2f}s")

            roc_data[attack_rate]['y_true'].extend(verifier.roc_labels)
            roc_data[attack_rate]['y_score'].extend(verifier.roc_scores)

    df_all_rounds = pd.DataFrame(all_round_results)
    df_all_rounds.to_csv('baseline_paperlike_noniid_targeted_all_round_results.csv', index=False)
    print(f"[CSV] Saved {len(df_all_rounds)} rows to 'baseline_paperlike_noniid_targeted_all_round_results.csv'")

    generate_accuracy_metric_plots(history, ATTACK_RATES, N_ROUNDS)

    print("\n" + "=" * 60)
    print(" FINAL RESULTS ACROSS ALL SEEDS")
    print("=" * 60)
    for ar in ATTACK_RATES:
        final_accs = history[ar]['acc'][:, -1]
        final_o_accs = history[ar]['o_acc'][:, -1]
        final_t_accs = history[ar]['t_acc'][:, -1]
        print(f"\nAttack Rate {ar * 100:.0f}%:")
        print(f"  Overall Accuracy    : {np.mean(final_accs):.4f} ± {np.std(final_accs):.4f}")
        print(f"  Other-Class Accuracy: {np.mean(final_o_accs):.4f} ± {np.std(final_o_accs):.4f}")
        print(f"  Target Accuracy     : {np.mean(final_t_accs):.4f} ± {np.std(final_t_accs):.4f}")
